# State preparation and the `PreparesKnownState` contract

A primitive is specified by its whole unitary; a block that *prepares* a state is specified by a single column of it, $U|0\ldots0\rangle$, and the other columns are free — so there is no unitary oracle to compare it against, and the column itself is the contract. A declaring block implements `target_statevector()` from its mathematical definition (never from its own commands); `prepared_statevector()` is what the built circuit actually leaves, and the conformance suite compares the two phase-exactly (`atol=1e-10`), or by infidelity against `error_bound` when `is_exact` is `False`. `state_qubits` says *where* the state lives when the block sizes itself larger than the state, and `ancilla_postselection` says *under what condition* — `None` meaning every ancilla returns to $|0\rangle$ with probability 1, so the block is control-safe.

This notebook shows the contract on an existing block, then walks through the blocks added with it, and ends with what they cost today.

In [ ]:
import numpy as np

import qarp.blocks as qb
import qarpx as qx
from qarp import PostSelection
from qarp.operators import JordanWigner
from qarp.resources import estimate

np.set_printoptions(precision=4, suppress=True, linewidth=120)

### The contract on an existing block

`CVOQRAMStateBlock` prepares an $n$-qubit state on a $2n$-qubit register — the first $n$ qubits are work ancillas that the construction restores. The declaration makes that explicit: `state_qubits` names the three qubits carrying the state, `ancilla_postselection` is `None` because the ancillas come back to $|0\rangle$ deterministically, and `prepared_statevector()` reduces the full register to those three qubits so it can be compared with `target_statevector()` directly, global phase included.

In [ ]:
block = qb.CVOQRAMStateBlock({(1, 0, 1): 0.6, (0, 1, 0): -0.8}).build()

print("n_qubits:            ", block.n_qubits)
print("state_qubits:        ", block.state_qubits)
print("ancilla_qubits:      ", block.ancilla_qubits)
print("ancilla_postselection:", block.ancilla_postselection)

prepared, probability = block.prepared_statevector()
print("probability of the declared condition:", probability)
print("prepared == target (phase-exact):", np.allclose(prepared, block.target_statevector(), atol=1e-10))
print("isinstance(block, PreparesKnownState):", isinstance(block, qb.PreparesKnownState))
print("declaring blocks:", sorted(qb.declaring_blocks()))

In [ ]:
def conforms(block) -> bool:
    # The check the conformance suite makes for every exact declaring block.
    prepared, _ = block.prepared_statevector()
    return bool(np.allclose(prepared, block.target_statevector(), atol=1e-10))


def nonzero_entries(psi: np.ndarray, n_qubits: int):
    # LSB throughout: index bit q is qubit q, so the tuple reads (qubit 0, qubit 1, ...).
    for idx in np.flatnonzero(np.abs(psi) > 1e-12):
        bits = tuple(int((idx >> q) & 1) for q in range(n_qubits))
        yield bits, psi[idx]

### `SparseStateBlock`

An ancilla-free preparation of a state with `s` nonzero amplitudes out of $2^n$: a magnitude tree of multi-controlled `Ry` rotations restricted to the `s - 1` branch points that carry support, then one multi-controlled phase per complex amplitude. The dict keys are LSB-first basis tuples (element `i` is qubit `i`), and the amplitudes are normalised internally.

In [ ]:
amplitudes = {(1, 0, 1): 0.6, (0, 1, 1): 0.8j}
sparse = qb.SparseStateBlock(n_qubits=3, amplitudes=amplitudes).build()

print("conforms:", conforms(sparse))
for bits, amp in nonzero_entries(sparse.prepared_statevector()[0], sparse.n_qubits):
    print(f"  |{bits}>  {amp:.4f}")

### `MultiONVStateBlock`

A CI-vector reference: a superposition of occupation-number vectors (abab order, `2*spatial` for α and `+1` for β) under a fermion-to-qubit mapping — the multi-determinant counterpart of `MappedONVStateBlock`. Each ONV goes through `mapping.encode_state` and the mapped bitstrings ride the same sparse construction as above. The sign convention matters when the coefficients come from elsewhere: pyscf's CI vectors use an αα…ββ… string order, and `qarp.operators.pyscf.onv_coefficients_from_civec` applies the per-determinant sign that brings them to qarp's ascending-index convention.

In [ ]:
onv_coefficients = {(1, 1, 0, 0): 0.9, (0, 0, 1, 1): -0.4}
multi = qb.MultiONVStateBlock(onv_coefficients, mapping=JordanWigner()).build()

print("conforms:", conforms(multi))
print(
    "Sign convention: the coefficient of ONV b multiplies a†_{p1} a†_{p2} ... a†_{pk} |vac>\n"
    "with p1 < p2 < ... < pk the occupied spin-orbital indices in abab order."
)
for bits, amp in nonzero_entries(multi.prepared_statevector()[0], multi.n_qubits):
    print(f"  |{bits}>  {amp:.4f}")

In [ ]:
try:
    import pyscf  # noqa: F401
    from pyscf import fci, gto, scf

    from qarp.operators.pyscf import onv_coefficients_from_civec

    mol = gto.M(atom="H 0 0 0; H 0 0 0.735", basis="sto-3g", verbose=0)
    mf = scf.RHF(mol).run()
    fci_energy, civec = fci.FCI(mf).kernel()

    coefficients = onv_coefficients_from_civec(civec, mol.nao, mol.nelec)
    h2 = qb.MultiONVStateBlock(coefficients).build()

    print(f"H2/sto-3g FCI energy: {fci_energy:.6f} Ha")
    for onv, c in coefficients.items():
        print(f"  ONV {onv}  coefficient {c.real:+.6f}")
    print("conforms:", conforms(h2))
except ImportError:
    print("pyscf not installed")

### `SlaterDeterminantBlock`

A single determinant of orbitals that are a *rotation* of the qubit orbitals: `Q` is a real `n_modes × n_occupied` matrix with orthonormal columns, and the block applies an `OrbitalRotationBlock` of its Gram–Schmidt completion to the reference $|1^M 0^{n-M}\rangle$. In the qubit basis that is a superposition of up to $\binom{N}{M}$ determinants with amplitudes given by the minors $\det Q[S, :]$. When `Q` is a slice of the identity the state is a single ONV and the block short-circuits to the `X` layer alone.

In [ ]:
rng = np.random.default_rng(7)
q_random, _ = np.linalg.qr(rng.normal(size=(4, 2)))
slater = qb.SlaterDeterminantBlock(q_random).build()

identity_slice = np.eye(4)[:, :2]
slater_identity = qb.SlaterDeterminantBlock(identity_slice).build()

for label, blk in [("random Q", slater), ("identity slice", slater_identity)]:
    n_nonzero = np.count_nonzero(np.abs(blk.prepared_statevector()[0]) > 1e-12)
    print(f"{label:15s} conforms: {conforms(blk)}   nonzero amplitudes: {n_nonzero}   commands: {len(blk.flatten())}")

### `CSFStateBlock`

A spin-adapted configuration state function, built classically by genealogical (Yamanouchi–Kotani) coupling of the open-shell electrons and handed to `MultiONVStateBlock` — deterministic, no projector and no postselection. Giving `S` selects the canonical coupling path; for two open-shell electrons and `S=0` that is the singlet $(\alpha\beta - \beta\alpha)/\sqrt{2}$, two determinants with opposite signs.

In [ ]:
csf = qb.CSFStateBlock(
    n_spatial_orbitals=2, core_orbitals=[], open_shell_orbitals=[0, 1], S=0, Ms=0
).build()

print("conforms:", conforms(csf))
print("coupling path:", csf.coupling_path)
for onv, c in csf.onv_coefficients.items():
    print(f"  ONV {onv}  coefficient {c.real:+.4f}")

### `LowRankStateBlock`

An ε-dial on dense synthesis: the amplitude tensor is Schmidt-decomposed across `cut`, only `max_schmidt_rank` terms are kept, and the circuit prepares the truncated state exactly. `target_statevector()` is still the *un-truncated* state the caller asked for, so a truncated block reports `is_exact = False` and `error_bound` equal to the discarded Schmidt weight — which is the infidelity to that target exactly. Leaving the rank at its full value makes the block exact again.

In [ ]:
rng = np.random.default_rng(0)
vector = rng.normal(size=16) + 1j * rng.normal(size=16)

for rank in (1, None):
    low_rank = qb.LowRankStateBlock(4, list(vector), cut=2, max_schmidt_rank=rank).build()
    prepared, _ = low_rank.prepared_statevector()
    infidelity = 1 - abs(np.vdot(low_rank.target_statevector(), prepared)) ** 2
    print(
        f"rank {low_rank.schmidt_rank} of {low_rank.full_schmidt_rank}:  "
        f"is_exact={low_rank.is_exact}  error_bound={low_rank.error_bound:.4f}  "
        f"measured infidelity={infidelity:.4f}  exact-conformance={conforms(low_rank)}"
    )

### `MPSStateBlock`

"I ran DMRG, give me a circuit": an open-boundary MPS, one tensor of shape `(chi_left, 2, chi_right)` per site, is right-canonicalised and each site becomes one unitary on a bond register plus that site's qubit. The bond register (`ceil(log2 chi)` qubits, here one) is an ancilla that the trivial right boundary returns to $|0\rangle$ deterministically, so `state_qubits` excludes it and `ancilla_postselection` is `None`. The two-site example below is $0.6|00\rangle + 0.8|11\rangle$.

In [ ]:
tensors = [
    np.array([[[0.6, 0.0], [0.0, 0.8]]]),
    np.array([[[1.0], [0.0]], [[0.0], [1.0]]]),
]
mps = qb.MPSStateBlock(tensors).build()

print("conforms:", conforms(mps))
print("n_qubits:", mps.n_qubits, " bond (ancilla) qubits:", mps.ancilla_qubits, " state_qubits:", mps.state_qubits)
print("prepared statevector on state_qubits:")
for bits, amp in nonzero_entries(mps.prepared_statevector()[0], len(mps.state_qubits)):
    print(f"  |{bits}>  {amp.real:.4f}")

### `UniformSuperpositionBlock`

$\frac{1}{\sqrt{M}}\sum_{j=0}^{M-1}|j\rangle$ for any integer `M`, not only a power of two, on the minimum $\lceil\log_2 M\rceil$ qubits. With `M = 11` the first eleven amplitudes are $1/\sqrt{11}$ and the remaining five of the 16 are exactly zero.

In [ ]:
uniform = qb.UniformSuperpositionBlock(11).build()
psi = uniform.prepared_statevector()[0]

print("conforms:", conforms(uniform), " n_qubits:", uniform.n_qubits)
print("amplitudes:", psi.real)
print("first 11 equal 1/sqrt(11):", np.allclose(psi[:11], 1 / np.sqrt(11)), "  rest zero:", np.allclose(psi[11:], 0))

### `PiecewiseLinearStateBlock`

The Woerner–Egger payoff operator: the domain register goes into uniform superposition and a flag qubit is rotated by an angle $\theta(x)$ that is linear in $x$ within each piece, so the flag's $|1\rangle$ amplitude is $\sin(\theta(x)/2)$. The block is deterministic — `state_qubits` are the domain qubits plus the flag, the comparator ancilla returns to $|0\rangle$, and `ancilla_postselection` is `None` — which is what makes it usable under `AmplitudeEstimationBlock`. Whoever wants the *postselected* profile $\propto \sum_x \sin(\theta(x)/2)|x\rangle$ applies a `PostSelection` on the flag afterwards; its success probability is the quantity amplitude estimation reads out.

In [ ]:
piecewise = qb.PiecewiseLinearStateBlock(
    n_domain_qubits=3, breakpoints=[3], slopes=[0.1, 0.3], intercepts=[0.0, -0.2]
).build()

prepared, probability = piecewise.prepared_statevector()
print("state_qubits:", piecewise.state_qubits, " ancilla_qubits:", piecewise.ancilla_qubits)
print("ancilla_postselection:", piecewise.ancilla_postselection, " probability:", round(probability, 12))
print("conforms:", conforms(piecewise))

# Postselect flag = 1 on the full register; the comparator ancilla is already |0>, so
# conditioning on it too just drops it from the output, leaving the 8-value domain profile.
condition = PostSelection({piecewise.flag_qubit: 1, **dict.fromkeys(piecewise.ancilla_qubits, 0)})
profile, success = condition.apply_statevector(piecewise.statevector(), piecewise.n_qubits)

x = np.arange(8)
theta = np.where(x < 3, 0.1 * x + 0.0, 0.3 * x - 0.2)
expected = np.sin(theta / 2) / np.linalg.norm(np.sin(theta / 2))
print("postselected profile:", profile.real)
print("matches sin(theta/2):", np.allclose(profile, expected))
print(f"success probability {success:.4f} = mean of sin^2(theta/2): {np.mean(np.sin(theta / 2) ** 2):.4f}")

### `QROMBlock`

Reversible classical data loading, $|l\rangle|0\rangle \mapsto |l\rangle|\mathrm{data}[l]\rangle$: a lookup table addressed by a quantum index register. It is an oracle valid for *any* index state, so it is not a `PreparesKnownState` declarer; applied after `H` on the index qubits it entangles index and data, $\sum_l |l\rangle|\mathrm{data}[l]\rangle$.

In [ ]:
table = {0: (1, 0, 1), 1: (0, 1, 1), 2: (1, 1, 0), 3: (0, 0, 1)}
qrom = qb.QROMBlock(index_qubits=2, data=table)

superpose_index = qb.SimpleBlock(qrom.n_qubits, name="H_index")
superpose_index.h([0, 1])
lookup = qb.CompositeBlock([superpose_index, qrom]).build()

entries = sorted(nonzero_entries(lookup.statevector(), lookup.n_qubits), key=lambda e: e[0][0] + 2 * e[0][1])
for bits, amp in entries:
    index = bits[0] + 2 * bits[1]
    print(f"  index {index} -> data {bits[2:]}  amplitude {amp.real:.4f}   table says {table[index]}")

### What they cost today

CNOT count after decomposition on `clifford_t_rz_gateset` at `O0`, against the dense `SynthesizedStateBlock` for the same vector. Kept to 8 qubits so the cell runs in seconds.

In [ ]:
def cnots(block) -> int:
    block.build()
    return estimate(block, gateset=qx.clifford_t_rz_gateset(), opt_level=qx.OptLevel.O0).final.n_2q


def dense_baseline(block) -> qb.SynthesizedStateBlock:
    n = len(block.state_qubits)
    psi = block.target_statevector()
    amplitudes = {tuple((i >> k) & 1 for k in range(n)): a for i, a in enumerate(psi) if abs(a) > 1e-14}
    return qb.SynthesizedStateBlock(n_qubits=n, amplitudes=amplitudes)


rng = np.random.default_rng(1)
sparse_amplitudes = {}
while len(sparse_amplitudes) < 2:
    address = tuple(int(b) for b in rng.integers(0, 2, size=8))
    sparse_amplitudes[address] = complex(rng.normal(), rng.normal())
shapes = [(1, 2, 2), *([(2, 2, 2)] * 6), (2, 2, 1)]

cases = [
    ("SparseStateBlock (n=8, s=2)", qb.SparseStateBlock(8, sparse_amplitudes)),
    ("UniformSuperpositionBlock (M=255)", qb.UniformSuperpositionBlock(2**8 - 1)),
    ("MPSStateBlock (N=8, chi=2)", qb.MPSStateBlock([rng.normal(size=s) + 1j * rng.normal(size=s) for s in shapes])),
]

print(f"{'block':36s} {'CNOTs':>8s} {'dense':>8s}")
for label, block in cases:
    print(f"{label:36s} {cnots(block):8d} {cnots(dense_baseline(block)):8d}")

These constructions are correct and polynomial, but they sit well above the scaling of the papers they cite: every multi-controlled rotation is a Barenco cascade over an ancilla-free `mcx`, so `SparseStateBlock` and `UniformSuperpositionBlock` cost ≈ O(s·n⁴) / O(L⁴) CNOTs rather than the cited O(s·n) / O(L), and only cross over below dense synthesis at n ≈ 16–20. `MPSStateBlock` is already linear in the chain length but synthesises a full unitary per site and carries a bond-register ancilla. That is the plan's Path B, decided at gate 2: land the blocks with honest cost statements, and rewrite each to its cited construction in a declared follow-up PR. `tests/test_blocks/test_state_preparation/test_state_prep_cost.py` pins today's numbers as a ceiling and carries the cited-scaling assertion as a strict `xfail` that flips the moment a rewrite lands.